In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy
import cooler
import pybedtools
from scipy import stats

In [47]:
import statsmodels.api as sm

In [48]:
import statsmodels.formula.api as smf

In [291]:
a1=pd.read_csv('.../main_info/processed_5col_files/snp_pairs.txt',sep='\t')
a2=pd.read_csv('.../main_info/processed_5col_files/snp_pairs2.txt',sep='\t')

a=pd.concat([a1,a2],axis=0)

a['lab']=a.chrom+':'+a.snp_pos.astype(str)
a['dist']=((a.start+a.end)/2).astype(int)-a.snp_pos

In [292]:
a=a.loc[abs(a.dist)<1000000]

In [293]:
a['snp_bin']=(a.snp_pos/100000).astype(int)
a['bin']=(a.start/100000).astype(int)

In [294]:
a['dist_bin']=a.bin-a.snp_bin

In [295]:
covs1=pd.read_csv('.../SNP_analysis/hg38_1k_GATC_bintolen.txt.gz',sep='\t')

In [296]:
covs1=pd.concat([covs1,covs1['bins'].str.split('-',expand=True)],axis=1)

In [297]:
covs2=pd.read_csv('.../SNP_analysis/hg38_100k_GATC_bintolen.txt.gz',sep='\t')
covs2=pd.concat([covs2,covs2['bins'].str.split('-',expand=True)],axis=1)

In [298]:
a.reset_index(drop=True,inplace=True)

In [299]:
a['idx']=a.index

In [300]:
a_bed1=pybedtools.BedTool.from_dataframe(a[['chrom','snp_pos','snp_pos','idx']])
a_bed2=pybedtools.BedTool.from_dataframe(a[['chrom','start','end','idx']])

In [301]:
expr_bed=pybedtools.BedTool('.../BA9_pos.merged.20kbp.bedGraph')

covs1_bed=pybedtools.BedTool.from_dataframe(covs1[[0,1,2,'gc','map','len']])
covs2_bed=pybedtools.BedTool.from_dataframe(covs2[[0,1,2,'gc','map','len']])

expr_int_bin=covs2_bed.intersect(expr_bed,wao=True).to_dataframe()
expr_int_bin['prop_expr']=(expr_int_bin.blockSizes/100000)*expr_int_bin.blockCount
expr_int_bin=expr_int_bin.groupby(['chrom','start','end']).sum().reset_index()

covs2['expr_cpm']=expr_int_bin.prop_expr

expr_snp=expr_bed.intersect(covs1_bed).to_dataframe()

covs1['expr']=expr_snp['name']

In [302]:
pc=['HC-91plus_frankenstein_compartments.bed',
 'HCM12plus_frankenstein_compartments.bed',
 'HC24plus_frankenstein_compartments.bed',
 'HC-318plus_frankenstein_compartments.bed',
 'SZ6plus_frankenstein_compartments.bed',
 'HC-2Mplus_frankenstein_compartments.bed',
 'SZ20plus_frankenstein_compartments.bed',
 'SZ08plus_frankenstein_compartments.bed',
 'SZ10plus_frankenstein_compartments.bed']

a1=pd.read_csv('.../PC1/HC-91plus_frankenstein_compartments.bed',sep='\t',header=None)

k=4
for i in pc[1:]:
    b=pd.read_csv('.../PC1/'+i,sep='\t',header=None)
    a1[k]=b[3]
    k+=1

a1=a1[[0,1,2,4,5,6,7,8,9,10,11]]
a1[12]=np.nanmean(a1[[4,5,6,7,8,9,10,11]],axis=1)
a1=a1[[0,1,2,12]]

pc1_bed=pybedtools.BedTool.from_dataframe(a1)
pc_int_bin=covs2_bed.intersect(pc1_bed,wao=True).to_dataframe()
pc_int_bin.loc[pc_int_bin.blockCount=='.','blockCount']='100'
pc_int_bin.blockCount=pc_int_bin.blockCount.astype(float)
pc_int_bin=pc_int_bin[['chrom','start','end','name','score','strand','blockCount']]
pc_int_bin=pc_int_bin.groupby(['chrom','start','end','name','score','strand']).mean().reset_index()
pc_int_bin.loc[pc_int_bin.blockCount==100,'blockCount']=np.nan
pc_int_bin.sort_values(['chrom','start','end'],inplace=True)

covs2[1]=covs2[1].astype(int)
covs2[2]=covs2[2].astype(int)
covs2.sort_values([0,1,2],inplace=True)
covs2['pc1']=list(pc_int_bin.blockCount)

pc_snp=covs1_bed.intersect(pc1_bed,wao=True).to_dataframe()
covs1['pc1']=pc_snp.blockCount

covs1[1]=covs1[1].astype(int)
covs1[2]=covs1[2].astype(int)

/tmp/ipykernel_3314276/1811724637.py:20: RuntimeWarning: Mean of empty slice
  a1[12]=np.nanmean(a1[[4,5,6,7,8,9,10,11]],axis=1)


In [303]:
covs1_bed=pybedtools.BedTool.from_dataframe(covs1[[0,1,2,'gc','map','len','expr','pc1']])
covs2_bed=pybedtools.BedTool.from_dataframe(covs2[[0,1,2,'gc','map','len','expr_cpm','pc1']])

In [304]:
a1=a_bed1.intersect(covs1_bed,wao=True).to_dataframe(names=np.arange(13))

In [305]:
a1=a1[[3,7,8,9,10,11]]

In [306]:
a2=a_bed2.intersect(covs2_bed,wao=True).to_dataframe(names=np.arange(13))

In [307]:
a2.drop_duplicates([0,1,2,3],keep='first',inplace=True)

In [308]:
a2.sort_values(3,inplace=True)

In [309]:
a2.index=a2[3]

In [310]:
a['gc_snp']=a1[7]
a['map_snp']=a1[8]
a['elen_snp']=a1[9]
a['expr_cpm']=a1[10]
a['pc1']=a1[11]

a['gc_bin']=a2[7]
a['map_bin']=a2[8]
a['elen_bin']=a2[9]
a['expr_cpm_bin']=a2[10]
a['pc1_bin']=a2[11]


In [311]:
#cont_gc=np.log(a.gc_snp*a.gc_bin)
#cont_gc[np.isinf(cont_gc)]=np.nan
#cont_gc=(cont_gc-np.nanmean(cont_gc))/np.nanstd(cont_gc)
cont_gc=a.gc_snp*a.gc_bin

In [312]:
#cont_map=np.log(a.map_snp*a.map_bin)
#cont_map[np.isinf(cont_map)]=np.nan
#cont_map[np.isnan(cont_map)]=np.min(cont_map)-np.min(cont_map)/20
#cont_map=(cont_map-np.nanmean(cont_map))/np.nanstd(cont_map)
###cont_map[np.isnan(cont_map)]=0
cont_map=a.map_snp*a.map_bin


In [313]:
#cont_elen=np.log(a.elen_snp*a.elen_bin)
#cont_elen[np.isinf(cont_elen)]=np.nan
#cont_elen=(cont_elen-np.nanmean(cont_elen))/np.nanstd(cont_elen)
cont_elen=a.elen_snp*a.elen_bin


In [314]:
a['gc']=cont_gc
a['map']=cont_map
a['elen']=cont_elen

In [323]:
a_with_pc1=a.loc[(a.pc1!='.')&(a.pc1_bin!='.')]
a_with_pc1.pc1=a_with_pc1.pc1.astype(float)
a_with_pc1.pc1_bin=a_with_pc1.pc1_bin.astype(float)

In [324]:
a_with_pc1['dif_pc1']=abs(a_with_pc1.pc1-a_with_pc1.pc1_bin)

In [325]:
bb=a_with_pc1.groupby(['lab','gt']).count().reset_index()
bb['lab2']=bb.lab+':'+bb['gt']
bb.index=bb.lab2

In [326]:
low=bb.loc[bb.chrom<30,'lab']

In [327]:
a_with_pc1=a_with_pc1.loc[~a_with_pc1.lab.isin(low)]

In [328]:
a_with_pc1=a_with_pc1[['chrom', 'snp_pos', 'start', 'end', 'gt', 'sample', 'lab', 'dist',
       'snp_bin', 'bin', 'dist_bin', 'idx', 'gc', 'elen','map','expr_cpm_bin','dif_pc1','expr_cpm']]# 'expr_cpm',

In [329]:
a_with_pc1.reset_index(drop=True,inplace=True)

In [330]:
labs=np.unique(a_with_pc1.lab)

In [331]:
n=21
k=10
size=len(a_with_pc1.groupby(['lab','gt']).count())
snp=np.zeros((size,n)).astype(str)
snp[:]=np.nan
dist=np.zeros((size,n))
counts=np.zeros((size,n))
genotype=np.zeros((size,n)).astype(str)
gc=np.zeros((size,n))
dif_pc=np.zeros((size,n))
expr_snp=np.zeros((size,n))
expr_bin=np.zeros((size,n))
mapp=np.zeros((size,n))
elen=np.zeros((size,n))
i=0
for l in labs:
    tmp=a_with_pc1.loc[a_with_pc1.lab==l]    
    gt=tmp.groupby(['gt']).count().index
    for g in gt:
        tmp2=tmp.loc[tmp['gt']==g]
        tmp3=tmp2.groupby('dist_bin').count()
        counts[i,tmp3.index+k]=tmp3.chrom
        snp[i,:]=np.repeat(l,n)
        dist[i,:]=np.arange(-k,k+1)
        genotype[i,:]=np.repeat(g,n)
        tmp4=tmp2[['dist_bin','gc','elen','map','dif_pc1','expr_cpm_bin','expr_cpm']].groupby('dist_bin').mean() #'expr_cpm',
        gc[i,tmp4.index+k]=tmp4['gc']
        mapp[i,tmp4.index+k]=tmp4['map']
        elen[i,tmp4.index+k]=tmp4['elen']
        dif_pc[i,tmp4.index+k]=tmp4['dif_pc1']
        expr_snp[i,tmp4.index+k]=tmp4['expr_cpm']
        expr_bin[i,tmp4.index+k]=tmp4['expr_cpm_bin']
        i+=1

In [332]:
df=pd.DataFrame({'counts':counts.flatten(),'snp':snp.flatten(),'dist':dist.flatten(),'gt':genotype.flatten(),'gc':gc.flatten(),'elen':elen.flatten(),'map':mapp.flatten(),
                'dif_pc1':dif_pc.flatten(),'expr_bin':expr_bin.flatten(),'expr_snp':expr_snp.flatten()}) #               




In [333]:
dg=df.copy()
dg=pd.concat([dg,dg.snp.str.split(':',expand=True)],axis=1)

In [334]:
dg[1]=dg[1].astype(int)
dg['agg_bin']=(((dg[1]/100000).astype(int))*100000+(dg.dist)*100000).astype(int)
dg['agg_bin']=dg['agg_bin']+1

In [335]:
agg_bin_bed=pybedtools.BedTool.from_dataframe(dg[[0,'agg_bin','agg_bin']])

In [336]:
t=agg_bin_bed.intersect(covs2_bed,wao=True).to_dataframe()

In [337]:
t=t[['thickStart','thickEnd','itemRgb','blockCount','blockSizes']]
t.columns=['gc_bin','map_bin','elen_bin','expr_cpm_bin','pc1_bin']
dg=pd.concat([dg,t],axis=1)

In [338]:
tt=dg.snp.str.split(':',expand=True)
tt[1]=tt[1].astype(int)
tt['ids']=tt.index
snp_bed=pybedtools.BedTool.from_dataframe(tt[[0,1,1]])

In [339]:
ttt=snp_bed.intersect(covs1_bed,wao=True).to_dataframe()

In [340]:
ttt.sort_values(['chrom','start'],inplace=True)

In [341]:
ttt.reset_index(drop=True,inplace=True)

In [342]:
dg.sort_values([0,1],inplace=True)

In [343]:
dg.reset_index(drop=True,inplace=True)

In [344]:
ttt=ttt[['thickStart','thickEnd','itemRgb','blockSizes','blockCount']]#
ttt.columns=['gc_snp','map_snp','elen_snp','pc1_snp_new','expr_snp_new']#'expr_snp_new',
dg=pd.concat([dg,ttt],axis=1)

In [345]:
dg=dg.loc[dg.gc_bin!='.']

In [346]:
dg.gc_bin=dg.gc_bin.astype(float)
#dg.map_bin=dg.map_bin.astype(float)
dg.elen_bin=dg.elen_bin.astype(float)

In [347]:
dg['gc_product']=dg.gc_snp*dg.gc_bin
#dg['map_product']=dg.map_bin#dg.map_snp
dg['elen_product']=dg.elen_snp*dg.elen_bin

In [348]:
dg.loc[dg.elen==0,'elen']=dg.loc[dg.elen==0,'elen_product']
#dg.loc[dg['map']==0,'map']=dg.loc[dg['map']==0,'map_product']
dg.loc[dg.gc==0,'gc']=dg.loc[dg.gc==0,'gc_product']

In [349]:
dg=dg.loc[dg.pc1_bin!='.']

In [350]:
dg.expr_cpm_bin=dg.expr_cpm_bin.astype(float)
dg.pc1_bin=dg.pc1_bin.astype(float)

In [351]:
dg.loc[dg.dif_pc1==0,'dif_pc1']=abs(dg.loc[dg.dif_pc1==0,'pc1_bin']-dg.loc[dg.dif_pc1==0,'pc1_snp_new'])

In [352]:
dg=dg[['snp','counts','dist','gt','gc','elen','dif_pc1','expr_cpm_bin','expr_snp_new']]#'map',,'expr_snp_new'

In [353]:
dg['lab_gt']=dg.snp+':'+dg['gt']

In [354]:
dh=dg.groupby(['lab_gt']).sum().reset_index()

In [355]:
dh.index=dh.lab_gt

In [356]:
dg['lib_size']=list(dh.loc[dg.lab_gt,'counts'])

In [357]:
dg.reset_index(drop=True,inplace=True)

In [358]:
dg_main=dg[['counts','dist','gc','elen','lib_size','dif_pc1','expr_cpm_bin','expr_snp_new']]#'map',,'gt'
dg_main.reset_index(drop=True,inplace=True)
dg_main.columns=['counts', 'distance', 'gc_content', 
                       'effective_length' ,'library_size','dif_pc1','expr_bin','expr_snp']#'mappability',,'gt'

In [359]:
dg_main.effective_length=np.log10(dg_main.effective_length+1)

In [360]:
for c in ['distance', 'gc_content', 
                       'effective_length' ,'library_size','dif_pc1','expr_bin','expr_snp']: #
    dg_main[c]=(dg_main[c]-np.mean(dg_main[c]))/np.std(dg_main[c])

In [361]:
dg_main

,counts,distance,gc_content,effective_length,library_size,dif_pc1,expr_bin,expr_snp
0,0.0,-1.653328,1.772704,-0.201063,1.163497,-0.112962,0.007545,0.137315
1,0.0,-1.487600,1.810948,-0.182919,1.163497,-0.119090,0.016096,0.137315
2,2.0,-1.321871,1.927802,-0.186550,1.163497,-0.117912,0.027160,0.137315
3,0.0,-1.156143,1.051186,-0.138432,1.163497,-0.105626,0.138049,0.137315
4,0.0,-0.990415,0.947079,-0.124566,1.163497,-0.103260,-0.014202,0.137315
...,...,...,...,...,...,...,...,...
22037,0.0,0.998325,-0.703871,0.208520,-0.703565,-0.118061,-0.089155,-0.237784
22038,2.0,1.164053,-0.457963,0.248830,-0.703565,-0.124526,-0.088960,-0.237784
22039,2.0,1.329781,-0.585496,0.243088,-0.703565,-0.129671,-0.087507,-0.237784
22040,1.0,1.495509,-0.398102,0.266565,-0.703565,-0.137532,-0.086936,-0.237784


In [406]:
from statsmodels.discrete.discrete_model import Poisson
from patsy import dmatrix

def fit_poisson_model_with_spline_and_libsize(df, spline_df=4):
    """
    Fit a hurdle zero-inflated negative binomial model to Hi-C data.
    Includes natural cubic spline for distance, GC content, mappability,
    and library size normalization as covariates.
    
    Parameters:
    df (pd.DataFrame): includes columns 'counts', 'distance', 'gc_content', 
                       'mappability', 'library_size'
    spline_df (int): degrees of freedom for spline basis on distance
    
    Returns:
    model_result: fitted model object
    """
    # Generate natural cubic spline basis for distance
    spline_basis = dmatrix(f"bs(distance, df={spline_df}, degree=3, include_intercept=False)", 
                           {"distance": df["distance"]}, return_type='dataframe')
    
    # Combine spline basis with other covariates including library size
    X = pd.concat([spline_basis, df[['gc_content', 'effective_length' ,'library_size','dif_pc1','expr_bin','expr_snp']]], axis=1)
    
    # Add intercept term
    X = sm.add_constant(X)
    
    y = df['counts']
    
    # Use same covariates for zero-inflation model
    model = Poisson(endog=y, exog=X)
    
    result = model.fit(maxiter=1000, disp=0)
    
    return result


def poisson_significance(df, model_result, two_sided=True, fdr_method="fdr_bh"):
    from scipy.stats import poisson
    
    mu = model_result.predict(which='mean')
    y = df['counts']
    pvalues = 1 - poisson.cdf(y - 1, mu)

    out = df.copy()

    p_upper = poisson.sf(y - 1, mu)  # P(X >= y)

    if two_sided:
        p_lower = poisson.cdf(y, mu)  # P(X <= y)
        pvals = 2 * np.minimum(p_lower, p_upper)
        pvals = np.clip(pvals, 0, 1)
    else:
        pvals = p_upper

    out["p_value"] = pvals

    reject, qvals, _, _ = multipletests(out["p_value"], alpha=0.05, method=fdr_method)
    out["q_value"] = qvals
    out["significant"] = reject
    out['p_upper']=p_upper
    out['p_lower']=p_lower

    return out


In [407]:
result=fit_poisson_model_with_spline_and_libsize(dg_main, spline_df=4)

In [408]:
result.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                          Poisson Regression Results                          
==============================================================================
Dep. Variable:                 counts   No. Observations:                22042
Model:                        Poisson   Df Residuals:                    22031
Method:                           MLE   Df Model:                           10
Date:                Sat, 16 May 2026   Pseudo R-squ.:                  0.3615
Time:                        10:12:06   Log-Likelihood:                -44357.
converged:                       True   LL-Null:                       -69471.
Covariance Type:            nonrobust   LLR p-value:                     0.000
============================================================================================================================
                                                               coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------------------------------------
Intercept                                                   -0.1236      0.027     -4.539      0.000      -0.177      -0.070
bs(distance, df=4, degree=3, include_intercept=False)[0]    -1.9192      0.050    -38.698      0.000      -2.016      -1.822
bs(distance, df=4, degree=3, include_intercept=False)[1]     6.0864      0.038    158.578      0.000       6.011       6.162
bs(distance, df=4, degree=3, include_intercept=False)[2]    -1.8306      0.046    -39.939      0.000      -1.920      -1.741
bs(distance, df=4, degree=3, include_intercept=False)[3]    -0.0569      0.038     -1.503      0.133      -0.131       0.017
gc_content                                                   0.0139      0.005      3.018      0.003       0.005       0.023
effective_length                                            -0.0036      0.005     -0.748      0.455      -0.013       0.006
library_size                                                 0.2928      0.004     83.400      0.000       0.286       0.300
dif_pc1                                                     -0.0720      0.007     -9.859      0.000      -0.086      -0.058
expr_bin                                                     0.0095      0.003      3.554      0.000       0.004       0.015
expr_snp                                                    -0.0022      0.005     -0.490      0.624      -0.011       0.007
============================================================================================================================
"""

In [481]:
pvals=poisson_significance(dg_main, result,two_sided=True)

In [482]:
pvals['snp']=dg.snp
pvals['gt']=dg['gt']
pvals['dist']=dg['dist']
pvals['lib_size']=dg['lib_size']

In [483]:
pvals['lab_dist']=pvals.snp+':'+pvals.dist.astype(str)

In [484]:
xx=pvals[['lab_dist','gt']].groupby(['lab_dist']).count().reset_index()
tri=xx.loc[xx['gt']>=3,'lab_dist']

In [485]:
len(np.unique(tri))/len(np.unique(pvals.lab_dist))

0.0821840166257321

In [486]:
pvals=pvals.loc[~pvals.lab_dist.isin(tri)]

In [487]:
len(np.unique(pvals.lab_dist))

9716

In [494]:
sign=pvals.loc[pvals.q_value<0.05]

In [495]:
sign.drop_duplicates('lab_dist',keep=False,inplace=True)

In [496]:
xx=pvals.loc[((pvals.q_value)>0.05)&(pvals.lab_dist.isin(sign.lab_dist))]

In [514]:
sign.loc[~sign.lab_dist.isin(xx.loc[xx.q_value<0.1,'lab_dist'])].to_csv('.../result_sign_of_poisson_glm_100k_strict_qval_0.1.txt',sep='\t',index=None)


In [515]:
sign_strict=sign.loc[~sign.lab_dist.isin(xx.loc[xx.q_value<0.1,'lab_dist'])]

In [517]:
not_sign=pvals.loc[~pvals.lab_dist.isin(sign_strict.lab_dist)]

not_sign.to_csv('.../no_sign_result_of_poisson_glm_100k_strict_qval_0.1.txt',sep='\t',index=None)


In [520]:
len(sign_strict)

251